# Laboratorium 6. Czy do przewidywania salda potrzebujemy i Limit, i Rating?

Zapisz kopię notebooka na swoim Dysku. Przed każdym doświadczeniem zapisz przewidywanie, a po obliczeniu dopisz, co w wynikach je potwierdza lub podważa.

## Przygotowanie

W danych *Credit* jeden wiersz opisuje klienta. Przewidujemy średnie saldo zadłużenia na karcie (`Balance`, USD). `Limit` to limit kredytowy w USD, `Rating` to ocena kredytowa w punktach, `Income` to dochód w tysiącach USD, a `Student` przyjmuje wartość 1 dla studenta. Uruchom komórkę. Do części 5 korzystaj tylko z 300 klientów w próbie uczącej.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data_path = Path("../dane/Credit.csv")
data_url = "https://rodakt.github.io/MAD/laby/dane/Credit.csv"
credit = pd.read_csv(data_path if data_path.is_file() else data_url)
features = ["Limit", "Rating", "Income", "Student"]
X = credit[features].copy()
X["Student"] = X["Student"].map({"No": 0, "Yes": 1})
X_tr, X_te, y_tr, y_te = train_test_split(
    X, credit["Balance"], test_size=100, random_state=707
)
print(f"Próba ucząca: {len(X_tr)} klientów. Próba testowa: {len(X_te)} klientów.")

## 1. Co wnosi druga cecha? (15 minut)

Narysuj ocenę kredytową względem limitu i oblicz korelację. Dopasuj model OLS przewidujący saldo na podstawie samego limitu. Ile dolarów różnicy przewiduje model dla klientów, których limity różnią się o 1000 USD?

<details><summary>Podpowiedź</summary>

`LinearRegression().fit(X_tr[["Limit"]], y_tr)` dopasowuje model z wyrazem wolnym. Nachylenia są w atrybucie `coef_`.

</details>

**Twoja odpowiedź**

### Co zmieni dodanie oceny kredytowej?

Przewidź, co stanie się z nachyleniem przy limicie oraz z predykcjami dla tych samych klientów. Dopasuj OLS z obiema cechami. Ponownie przelicz nachylenie na 1000 USD. Co teraz trzeba utrzymać na stałym poziomie? Policz pierwiastek ze średniej kwadratów różnic między predykcjami modeli. Porównaj go z odchyleniem standardowym salda.

<details><summary>Podpowiedź</summary>

Oba wektory predykcji powinny odpowiadać tym samym wierszom w tej samej kolejności.

</details>

**Twoja odpowiedź**

## 2. Czy jednostka limitu wpływa na predykcję ridge? (15 minut)

Pozostań przy dwóch cechach. Przygotuj drugą wersję danych z limitem w tysiącach USD. Przewidź, czy sama zmiana zapisu wpłynie na predykcje. Dopasuj osobny ridge z `alpha=100` w każdej wersji i porównaj predykcje przez RMS różnic. Powtórz doświadczenie, poprzedzając ridge standaryzacją obu cech. Wyjaśnij wynik przez zmianę współczynnika potrzebną do opisania tej samej zależności po zmianie jednostek.

<details><summary>Podpowiedź</summary>

`make_pipeline(StandardScaler(), Ridge(alpha=100))` tworzy potok. Dopasowujesz go i przewidujesz tak samo jak pojedynczy model.

</details>

**Twoja odpowiedź**

## 3. Które współczynniki reagują na karę? (20 minut)

Użyj odtąd wszystkich czterech cech. Dopasuj ridge z `alpha=1` po standaryzacji. Odczytaj cztery współczynniki w kolejności kolumn. Co w dolarach oznacza współczynnik przy standaryzowanym limicie? Przewidź, co zrobi z nim silniejsza kara.

<details><summary>Podpowiedź</summary>

`potok[-1].coef_` odczytuje współczynniki ostatniego kroku dopasowanego potoku.

</details>

**Twoja odpowiedź**

### Od jednego dopasowania do dwóch ścieżek

Napisz pętlę po 31 wartościach `alpha` rozmieszczonych geometrycznie: od 0,001 do 100 000 dla ridge i od 0,001 do 1000 dla LASSO. Dla każdej kary zapisz współczynniki. Narysuj dla każdej metody osobny wykres z czterema liniami i logarytmiczną osią kary. Sprawdź w liczbach, przy jakiej pierwszej badanej karze LASSO zeruje jedną z cech pary `Limit`, `Rating`. Czy druga ma wtedy niezerowy współczynnik? Co to mówi o wymienności tych cech?

<details><summary>Podpowiedź</summary>

`np.geomspace(poczatek, koniec, 31)` tworzy siatkę. Do dokładnego odczytu zer użyj `Lasso(max_iter=100000, tol=1e-8)`. Biblioteczne parametry alpha obu metod mają różne skale.

</details>

**Twoja odpowiedź**

## 4. Którą procedurę wybierzesz? (30 minut)

Ustal wspólne pięć części przez `KFold(n_splits=5, shuffle=True, random_state=708)`. Porównaj średnie walidacyjne MSE trzech modeli OLS: wszystkie cztery cechy, bez limitu, bez oceny. Dochód i status studenta zostają w każdym modelu. Które uproszczenie kosztuje najwięcej według CV?

<details><summary>Podpowiedź</summary>

`cross_val_score(model, X, y, cv=cv, scoring="neg_mean_squared_error")` zwraca pięć wartości przeciwnego MSE.

</details>

**Twoja odpowiedź**

### Dobierz karę i zapisz decyzję

Dla ridge i LASSO wykorzystaj siatki z części 3. Oceń każdy potok na tych samych podziałach co OLS. Dlaczego standaryzacja powinna być krokiem potoku przekazanego do CV? Narysuj krzywe średniego MSE względem kary. Dla każdej metody odczytaj minimum i pokaż obok niego wyniki trzech OLS. Zapisz metodę, cechy i ewentualną karę zwycięzcy według najmniejszego MSE CV. Jak duża jest jego przewaga?

<details><summary>Podpowiedź</summary>

Pętla ma tę samą konstrukcję co przy ścieżkach. Tym razem zapisujesz błędy z walidacji. Przydadzą się też w opcjonalnym zadaniu.

</details>

**Twoja odpowiedź**

## 5. Jak dobrze przewidujesz saldo nowego klienta? (10 minut)

Dopasuj wybraną procedurę na wszystkich 300 klientach uczących. Teraz wykonaj predykcję na odłożonych 100 klientach i podaj RMSE w dolarach. Zapisz dwa wnioski. Jakiej wielkości błędu predykcji można oczekiwać według tej oceny? Na ile wyniki CV i ścieżki uzasadniają pozostawienie obu cech w modelu?

<details><summary>Podpowiedź</summary>

Użyj tych samych kolumn i ustawień co przy wyborze. Pierwiastek z MSE przywraca jednostkę salda.

</details>

**Twoja odpowiedź**

## Opcjonalnie. Jak silną karę dopuszcza reguła 1 SE?

Wróć do uczących wyników LASSO. Dla kary o najmniejszym średnim MSE oblicz błąd standardowy jako odchylenie pięciu MSE (z dzielnikiem 4) podzielone przez pierwiastek z 5. Wybierz największą karę, której średni błąd mieści się w granicy minimum plus 1 SE. Porównaj liczbę niezerowych współczynników przy obu karach.

<details><summary>Podpowiedź</summary>

Ustal próg raz, na podstawie wyników przy minimum.

</details>

**Twoja odpowiedź**